In [1]:
import os
import sys
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

print("DeepShield Research Environment")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Device: CPU")

DeepShield Research Environment
Python: 3.14.10 (main, Nov  4 2026, 18:21:50) [GCC 16.2.1 20260810]
PyTorch: 2.15.0+cpu
Device: CPU


In [3]:
PROJECT_ROOT = Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_RAW)
print("Processed data:", DATA_PROCESSED)

Project: /home/darling/Projects/DeepShield
Raw data: /home/darling/Projects/DeepShield/data/raw
Processed data: /home/darling/Projects/DeepShield/data/processed


In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(
    "/home/darling/Projects/DeepShield/data/raw/CIC-IDS2017/"
    "MachineLearningCSV/MachineLearningCVE"
)

csv_files = sorted(DATA_DIR.glob("*.csv"))

print("Total CSV files:", len(csv_files))

for file in csv_files:
    df = pd.read_csv(file, nrows=5)

    # Remove accidental whitespace around column names
    df.columns = df.columns.str.strip()

    print("\n" + "=" * 65)
    print("FILE:", file.name)
    print("NUMBER OF COLUMNS:", len(df.columns))
    print("LABEL COLUMN EXISTS:", "Label" in df.columns)
    print("FIRST 8 COLUMNS:", df.columns[:8].tolist())

    if "Label" in df.columns:
        print("SAMPLE LABELS:", df["Label"].unique())

Total CSV files: 8

FILE: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
SAMPLE LABELS: <StringArray>
['BENIGN']
Length: 1, dtype: str

FILE: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min']
SAMPLE LABELS: <StringArray>
['BENIGN']
Length: 1, dtype: str

FILE: Friday-WorkingHours-Morning.pcap_ISCX.csv
NUMBER OF COLUMNS: 79
LABEL COLUMN EXISTS: True
FIRST 8 COLUMNS: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of F

In [4]:
from collections import Counter

label_counts = Counter()

for file in csv_files:
    print("Scanning:", file.name)

    for chunk in pd.read_csv(
        file,
        usecols=lambda col: col.strip() == "Label",
        chunksize=100_000
    ):
        labels = chunk.iloc[:, 0].astype("string").str.strip()

        label_counts.update(
            labels.value_counts().to_dict()
        )

print("\n========== DATASET LABEL DISTRIBUTION ==========")

for label, count in label_counts.most_common():
    print(f"{label:<25} {count:>12,}")

print("\nTotal records:", sum(label_counts.values()))
print("Total unique labels:", len(label_counts))

Scanning: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Scanning: Friday-WorkingHours-Morning.pcap_ISCX.csv
Scanning: Monday-WorkingHours.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Scanning: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Scanning: Tuesday-WorkingHours.pcap_ISCX.csv
Scanning: Wednesday-workingHours.pcap_ISCX.csv

========== DATASET LABEL DISTRIBUTION ==========
BENIGN                       2,273,097
DoS Hulk                       231,073
PortScan                       158,930
DDoS                           128,027
DoS GoldenEye                   10,293
FTP-Patator                      7,938
SSH-Patator                      5,897
DoS slowloris                    5,796
DoS Slowhttptest                 5,499
Bot                              1,966
Web Attack � Brute Force         1,507
Web Attack � XSS                   652
Infiltration                        3

In [5]:
import pandas as pd

label_df = pd.DataFrame(
    label_counts.items(),
    columns=["Label", "Record_Count"]
)

label_df["Percentage"] = (
    label_df["Record_Count"] / label_df["Record_Count"].sum()
) * 100

label_df = label_df.sort_values(
    by="Record_Count",
    ascending=False
)

label_df.to_csv(
    "../reports/class_distribution.csv",
    index=False
)

display(label_df)

,Label,Record_Count,Percentage
1,BENIGN,2273097,80.300366
10,DoS Hulk,231073,8.162981
2,PortScan,158930,5.614427
0,DDoS,128027,4.522735
13,DoS GoldenEye,10293,0.363615
8,FTP-Patator,7938,0.280421
9,SSH-Patator,5897,0.208320
11,DoS slowloris,5796,0.204752
12,DoS Slowhttptest,5499,0.194260
3,Bot,1966,0.069452
